# House-Edge — a quantitative teardown 🔬
### Full-notional vs idealized financing · the cost-of-carry · the financing-markup sweep · drawdown-vs-return · the volatility tax

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Free leverage?: Busted](https://img.shields.io/badge/Free_leverage%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its number.* The steelman: a vol-targeted contrarian dip-buyer with a trend gate, levered, beats buy-and-hold. We charge it the financing a real levered position pays and show the return edge is an accounting artefact while the drawdown protection is real.

> ⚠️ **Not investment advice.** The core executes on a synthetic GARCH-with-bear-regimes control; the real S&P run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from house_edge import data, strategy, costs, extension

# Offline synthetic control: an equity index with GARCH vol-clustering + persistent bear regimes.
# The real S&P 500 verdict is in ../docs/results.md.
price, rate, truth = data.synthetic_market(seed=30)
e = strategy.exposure(price)
print(f"synthetic control: {truth.n_days} days, {truth.n_crashes} bear regimes, seed {truth.seed}")


synthetic control: 6048 days, 9 bear regimes, seed 30


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — does the risk control work? | 🟢 `REAL` | maxDD **-27.8%** vs buy-and-hold **-55.7%** (S&P, 1990-2026); robust on the synthetic control. |
| **Tradability** — beats buy-and-hold once financed honestly? | 🔴 `MIRAGE` | CAGR **5.3%** vs **10.6%**, Sharpe **0.42** vs **0.65**, negative edge at every markup. |
| **Free leverage?** | ⚪ `Busted` | Idealized − honest = **1.55 pts/yr**, the hidden carry. |

> **In one sentence:** a vol-targeted contrarian book halves the drawdown but, financed at its true full-notional cost, makes half the return of the total-return index it levers — risk control real, return edge a mirage.

*(This notebook executes on the synthetic control; the real numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Exposure $e_t = \mathrm{clip}(\sigma^\star/\hat\sigma_t,\,0,\,2)\cdot m_t\cdot g_t$, where $\sigma^\star$ is the target vol, $\hat\sigma_t$ trailing 20-day realised vol, $m_t\in\{1.3,1,0.6\}$ the RSI(2) contrarian multiplier, and $g_t\in\{0,1\}$ the 200-day trend gate. The net return under cost model $c$ is $r^c_t = e_{t-1}r_t + \text{div} - \text{fin}^c_t - \text{spread}_t + \text{cash}^c_t$. The claim under test: $\mathrm{CAGR}(r^{\text{honest}}) > \mathrm{CAGR}(\text{buy\&hold})$.

In [2]:
s_bh = strategy.summary(costs.buy_and_hold(price))
s_hon = strategy.summary(costs.net_returns(e, price, rate, mode='honest'))
print(f"synthetic: strat CAGR {s_hon['cagr']*100:.1f}% vs buy&hold {s_bh['cagr']*100:.1f}% "
      f"-- claim is {'TRUE' if s_hon['cagr']>s_bh['cagr'] else 'FALSE'} here too")

synthetic: strat CAGR 2.0% vs buy&hold 7.8% -- claim is FALSE here too


## Beat 2 · So what?

Two literatures collide. Vol-targeting (Moreira-Muir 2017) and short-horizon reversal (Lehmann 1990) are real and raise *risk-adjusted* returns. But the cost-of-carry (Hull) makes leverage fund its **whole** notional, and constant leverage $k$ pays a volatility tax $\approx \tfrac12(k^2-k)\sigma^2$. The empirical question is which dominates once you stop financing leverage for free — beats 4-6 show the carry + tax beat the timing edge.

## Beat 3 · Pre-registered protocol

1. **Drawdown** (`extension.drawdown_protection`): strat maxDD vs buy-and-hold. Real ⇔ strictly smaller.
2. **Return** (`strategy.summary` on `costs.net_returns`, honest): CAGR/Sharpe vs buy-and-hold.
3. **House edge** (`costs.house_edge`): idealized − honest CAGR > 0.
4. **Robustness** (`extension.financing_sweep`): edge vs buy-and-hold across the markup.

**Mirage line:** honest CAGR < buy-and-hold at every markup.

## Beat 4 · The teardown

### 4a · Drawdown protection (real) vs the return shortfall (mirage)

In [3]:
rows = {'buy & hold': strategy.summary(costs.buy_and_hold(price)),
        'idealized':  strategy.summary(costs.net_returns(e, price, rate, mode='idealized')),
        'honest':     strategy.summary(costs.net_returns(e, price, rate, mode='honest'))}
tbl = pd.DataFrame(rows).T[['cagr','sharpe','max_drawdown','calmar']]
display((tbl*[100,1,100,1]).round(2).rename(columns={'cagr':'CAGR%','max_drawdown':'maxDD%'}))

,CAGR%,sharpe,maxDD%,calmar
buy & hold,7.800,0.510,-44.270,0.180
idealized,2.980,0.290,-33.430,0.090
honest,2.020,0.220,-35.220,0.060


> 💡 **In plain words.** Drawdown shrinks (real risk control), but CAGR shrinks *too* — and the idealized→honest step alone costs a chunk of return. On the real S&P the honest book makes **5.3%** vs **10.6%**, same Calmar **0.19** — drawdown bought 1:1 with return.

### 4b · The house-edge identity

In [4]:
he = costs.house_edge(e, price, rate)
print(f"idealized CAGR {he['cagr_idealized']*100:.2f}%  -  honest CAGR {he['cagr_honest']*100:.2f}%  "
      f"=  house edge {he['house_edge_ann']*100:.2f} pts/yr   (avg exposure {he['avg_exposure']:.2f}x)")
print('On the real S&P (../docs/results.md): house edge 1.55 pts/yr, avg exposure 0.97x.')

idealized CAGR 2.98%  -  honest CAGR 2.02%  =  house edge 0.95 pts/yr   (avg exposure 0.70x)
On the real S&P (../docs/results.md): house edge 1.55 pts/yr, avg exposure 0.97x.


## Beat 5 · The verdict

- **Real risk control** (4a): maxDD -27.8% vs -55.7%.
- **No return edge** (4a): honest CAGR 5.3% < buy-and-hold 10.6%, lower Sharpe.
- **Hidden carry** (4b): house edge 1.55 pts/yr.

> **Signal `REAL` · Tradability `MIRAGE` · Free leverage? `Busted`.**

## Beat 6 · Could you trade it?

- **Break-even is unreachable:** the edge vs buy-and-hold is negative *at a zero markup* — volatility drag alone sinks it. No cost assumption rescues the return claim.
- **Capacity isn't the constraint; carry is.** Unlike a crowded alpha, this fails on cost, not scale.
- **Honest use case:** a drawdown-reduction overlay for a levered book that would otherwise get liquidated — sized for survival, not for beating the index.

## Beat 7 · Going further

### 7a · Worked complement — the financing-markup sweep
Is the shortfall a markup artefact? Sweep the broker spread over the T-bill.

In [5]:
sw = extension.financing_sweep(price, rate)
display((sw*[100,1,100]).round(2).rename(columns={'strat_cagr':'strat_CAGR%','edge_vs_bh_cagr':'edge_vs_bh_CAGR%'}))
print('Real S&P (../docs/results.md): edge vs buy&hold negative at every markup (-2.7% to -7.9%).')

,strat_CAGR%,strat_sharpe,edge_vs_bh_CAGR%
markup,,,
0.000,3.830,0.360,-3.970
0.010,3.100,0.300,-4.700
0.025,2.020,0.220,-5.780
0.050,0.250,0.080,-7.550


Real S&P (../docs/results.md): edge vs buy&hold negative at every markup (-2.7% to -7.9%).


**The result.** The levered book's edge over buy-and-hold is **negative across the whole markup range, including a zero markup** — funding leverage at the pure risk-free rate still leaves it behind the index it levers. The shortfall is structural (volatility drag + full-notional carry), not a fee assumption. Full real run in [`../docs/results.md`](../docs/results.md).

### 7b · Other forks
- **Levered constant-tilt vs timing overlay** — does a static 1.3× risk-parity book (no timing) dominate the vol-targeted timing one, net of the same honest carry?
- **Utility-priced drawdown** — at what risk aversion does the −28% vs −56% drawdown justify the −5pts/yr CAGR? Make the trade-off explicit.
- **Term-structure of the markup** — retail CFD swaps are often 2.5–3.5% over the bill; institutional futures roll near it. Price both.

PRs welcome — add the constant-tilt benchmark or the utility model.